In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import accuracy_score, classification_report

In [4]:
data_multi = pd.read_csv("/content/drive/MyDrive/ERP/data_multi.csv")

In [5]:
data_multi

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,LOS,GENDER,AGE,Heart Rate,O2 Saturation,Respiratory Rate,...,Height,Weight,GCS Total,Lactic Acid,FiO2,PaO2,ADMISSION_TYPE,INSURANCE,HOSPITAL_EXPIRE_FLAG,AgeGroup
0,3,145834,211552,2101-10-21 11:00:00,6.0646,1,77.0,88.0,98.0,9.0,...,175.3,269.240000,15.0,1.601220,0.40,80.772097,1,2,0,61-80
1,3,145834,211552,2101-10-24 21:00:00,6.0646,1,77.0,86.0,98.0,17.0,...,175.3,269.240000,15.0,1.549035,0.40,80.772097,1,2,0,61-80
2,3,145834,211552,2101-10-24 20:00:00,6.0646,1,77.0,85.0,99.0,17.0,...,175.3,269.240000,15.0,0.730116,0.40,105.000000,1,2,0,61-80
3,3,145834,211552,2101-10-24 19:00:00,6.0646,1,77.0,84.0,100.0,18.0,...,175.3,269.240000,15.0,1.471546,0.40,105.000000,1,2,0,61-80
4,3,145834,211552,2101-10-24 18:00:00,6.0646,1,77.0,88.0,100.0,17.0,...,175.3,269.240000,15.0,0.612001,0.40,105.000000,1,2,0,61-80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3072757,99999,113369,246512,2117-12-31 14:00:00,1.1242,0,64.0,98.0,97.0,18.0,...,161.3,75.416445,4.0,0.764668,0.21,69.524495,0,2,0,61-80
3072758,99999,113369,246512,2117-12-31 13:00:00,1.1242,0,64.0,94.0,97.0,15.0,...,161.3,75.416445,4.0,0.823122,0.21,69.524495,0,2,0,61-80
3072759,99999,113369,246512,2118-01-01 13:00:00,1.1242,0,64.0,86.0,97.0,15.0,...,161.3,75.416445,4.0,1.818850,0.21,69.524495,0,2,0,61-80
3072760,99999,113369,246512,2118-01-01 00:00:00,1.1242,0,64.0,100.0,97.0,21.0,...,161.3,75.416445,4.0,1.773556,0.21,69.524495,0,2,0,61-80


In [6]:
clinical = pd.read_csv("/content/drive/MyDrive/ERP/NOTEEVENTS.csv")

/tmp/ipython-input-6-2522486148.py:1: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical = pd.read_csv("/content/drive/MyDrive/ERP/NOTEEVENTS.csv")


In [7]:
clinical = clinical.drop(columns = ['CHARTTIME','STORETIME', "CGID","ISERROR"], axis = 1)
clinical

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,DESCRIPTION,TEXT
0,174,22532,167853.0,2151-08-04,Discharge summary,Report,Admission Date: [**2151-7-16**] Dischar...
1,175,13702,107527.0,2118-06-14,Discharge summary,Report,Admission Date: [**2118-6-2**] Discharg...
2,176,13702,167118.0,2119-05-25,Discharge summary,Report,Admission Date: [**2119-5-4**] D...
3,177,13702,196489.0,2124-08-18,Discharge summary,Report,Admission Date: [**2124-7-21**] ...
4,178,26880,135453.0,2162-03-25,Discharge summary,Report,Admission Date: [**2162-3-3**] D...
...,...,...,...,...,...,...,...
2083175,2070657,31097,115637.0,2132-01-21,Nursing/other,Report,NPN\n\n\n#1 Infant remains in RA with O2 sats...
2083176,2070658,31097,115637.0,2132-01-21,Nursing/other,Report,"Neonatology\nDOL #5, CGA 36 weeks.\n\nCVR: Con..."
2083177,2070659,31097,115637.0,2132-01-21,Nursing/other,Report,Family Meeting Note\nFamily meeting held with ...
2083178,2070660,31097,115637.0,2132-01-21,Nursing/other,Report,NPN 1800\n\n\n#1 Resp: [**Known lastname 2243*...


In [8]:
agg_notes = clinical.dropna(subset=['TEXT']).reset_index(drop=True)
texts = agg_notes['TEXT'].astype(str).tolist()

In [9]:
# Filter for relevant admissions
relevant_hadm_ids = set(data_multi['HADM_ID'])
clinical = clinical[clinical['HADM_ID'].isin(relevant_hadm_ids)]

# Aggregate notes (all text per HADM_ID)
agg_notes = clinical.groupby(['SUBJECT_ID', 'HADM_ID', 'CHARTDATE',  'CATEGORY', 'DESCRIPTION'])['TEXT'].apply(lambda notes: ' '.join(notes)).reset_index()


In [10]:
agg_notes

,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,DESCRIPTION,TEXT
0,3,145834.0,2101-10-20,ECG,Report,Sinus rhythm\nInferior/lateral T changes are n...
1,3,145834.0,2101-10-20,Radiology,CHEST (PORTABLE AP),[**2101-10-20**] 10:23 PM\n CHEST (PORTABLE AP...
2,3,145834.0,2101-10-20,Radiology,CT ABDOMEN W/O CONTRAST,[**2101-10-20**] 5:49 PM\n CT ABDOMEN W/O CONT...
3,3,145834.0,2101-10-21,ECG,Report,Normal sinus rhythm\nFirst degree AV block\n- ...
4,3,145834.0,2101-10-21,Echo,Report,PATIENT/TEST INFORMATION:\nIndication: S/P Car...
...,...,...,...,...,...,...
567003,99999,113369.0,2117-12-31,Physician,Physician Surgical Admission Note,Chief Complaint: Respiratory distress\n HPI:...
567004,99999,113369.0,2118-01-01,Nursing,Nursing Progress Note,"Respiratory failure, acute (not ARDS/[**Doctor..."
567005,99999,113369.0,2118-01-01,Nursing,Nursing Transfer Note,63 yo F with history of neurogenic claudicatio...
567006,99999,113369.0,2118-01-01,Physician,Intensivist Note,"SICU\n HPI:\n 63 F POD1 s/p PLIF, with res..."


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectors for the notes column
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_features = tfidf.fit_transform(agg_notes['TEXT']).toarray()

In [12]:
tfidf_features

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.2721951 , 0.        , 0.        , ..., 0.        , 0.        ,
        0.12309659],
       [0.13007188, 0.        , 0.        , ..., 0.        , 0.        ,
        0.03921551],
       ...,
       [0.09827251, 0.05802802, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.04926237, 0.04600217, ..., 0.05095687, 0.        ,
        0.        ],
       [0.04369268, 0.06879916, 0.0160615 , ..., 0.        , 0.        ,
        0.01317295]])

In [13]:
tfidf_feature_names = [f"tfidf_{i}" for i in range(tfidf_features.shape[1])]

In [14]:
import pandas as pd

tfidf_df = pd.DataFrame(tfidf_features, columns=tfidf_feature_names, index=agg_notes.index)


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(tfidf_df)
tfidf_scaled_df = pd.DataFrame(X_scaled, columns=tfidf_df.columns, index=tfidf_df.index)


In [16]:
agg_notes = pd.concat([agg_notes, tfidf_scaled_df], axis=1)

In [17]:
agg_notes

,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,DESCRIPTION,TEXT,tfidf_0,tfidf_1,tfidf_2,tfidf_3,...,tfidf_90,tfidf_91,tfidf_92,tfidf_93,tfidf_94,tfidf_95,tfidf_96,tfidf_97,tfidf_98,tfidf_99
0,3,145834.0,2101-10-20,ECG,Report,Sinus rhythm\nInferior/lateral T changes are n...,-0.520320,-0.447449,-0.442609,-0.486031,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
1,3,145834.0,2101-10-20,Radiology,CHEST (PORTABLE AP),[**2101-10-20**] 10:23 PM\n CHEST (PORTABLE AP...,2.099285,-0.447449,-0.442609,-0.486031,...,2.808442,0.482978,-0.210975,-0.449029,-0.370967,-0.358055,0.175542,-0.429672,-0.251045,0.947881
2,3,145834.0,2101-10-20,Radiology,CT ABDOMEN W/O CONTRAST,[**2101-10-20**] 5:49 PM\n CT ABDOMEN W/O CONT...,0.731491,-0.447449,-0.442609,-0.486031,...,-0.464806,-0.565398,-0.210975,0.510729,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.191577
3,3,145834.0,2101-10-21,ECG,Report,Normal sinus rhythm\nFirst degree AV block\n- ...,-0.520320,-0.447449,-0.442609,-0.486031,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
4,3,145834.0,2101-10-21,Echo,Report,PATIENT/TEST INFORMATION:\nIndication: S/P Car...,0.606697,0.008939,-0.198528,-0.254620,...,-0.464806,0.517090,-0.210975,1.106314,-0.370967,-0.358055,-0.446205,-0.429672,2.381485,-0.724288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567003,99999,113369.0,2117-12-31,Physician,Physician Surgical Admission Note,Chief Complaint: Respiratory distress\n HPI:...,-0.520320,1.605227,-0.442609,2.636398,...,1.224191,0.246046,-0.210975,0.716871,-0.370967,4.768429,-0.446205,0.765088,-0.251045,-0.724288
567004,99999,113369.0,2118-01-01,Nursing,Nursing Progress Note,"Respiratory failure, acute (not ARDS/[**Doctor...",-0.520320,1.922412,-0.442609,-0.486031,...,1.485180,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
567005,99999,113369.0,2118-01-01,Nursing,Nursing Transfer Note,63 yo F with history of neurogenic claudicatio...,0.425454,0.510033,-0.442609,0.970444,...,-0.464806,0.191607,-0.210975,0.638653,-0.370967,2.830315,-0.446205,-0.429672,-0.251045,-0.724288
567006,99999,113369.0,2118-01-01,Physician,Intensivist Note,"SICU\n HPI:\n 63 F POD1 s/p PLIF, with res...",-0.520320,0.365397,-0.007890,1.574738,...,0.872858,0.077255,-0.210975,0.474349,2.767061,3.702053,-0.065075,0.516563,-0.251045,-0.724288


In [18]:
agg_notes.drop(columns = "TEXT", axis =1)

,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,DESCRIPTION,tfidf_0,tfidf_1,tfidf_2,tfidf_3,tfidf_4,...,tfidf_90,tfidf_91,tfidf_92,tfidf_93,tfidf_94,tfidf_95,tfidf_96,tfidf_97,tfidf_98,tfidf_99
0,3,145834.0,2101-10-20,ECG,Report,-0.520320,-0.447449,-0.442609,-0.486031,-0.316164,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
1,3,145834.0,2101-10-20,Radiology,CHEST (PORTABLE AP),2.099285,-0.447449,-0.442609,-0.486031,-0.316164,...,2.808442,0.482978,-0.210975,-0.449029,-0.370967,-0.358055,0.175542,-0.429672,-0.251045,0.947881
2,3,145834.0,2101-10-20,Radiology,CT ABDOMEN W/O CONTRAST,0.731491,-0.447449,-0.442609,-0.486031,-0.316164,...,-0.464806,-0.565398,-0.210975,0.510729,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.191577
3,3,145834.0,2101-10-21,ECG,Report,-0.520320,-0.447449,-0.442609,-0.486031,-0.316164,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
4,3,145834.0,2101-10-21,Echo,Report,0.606697,0.008939,-0.198528,-0.254620,-0.316164,...,-0.464806,0.517090,-0.210975,1.106314,-0.370967,-0.358055,-0.446205,-0.429672,2.381485,-0.724288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567003,99999,113369.0,2117-12-31,Physician,Physician Surgical Admission Note,-0.520320,1.605227,-0.442609,2.636398,-0.316164,...,1.224191,0.246046,-0.210975,0.716871,-0.370967,4.768429,-0.446205,0.765088,-0.251045,-0.724288
567004,99999,113369.0,2118-01-01,Nursing,Nursing Progress Note,-0.520320,1.922412,-0.442609,-0.486031,-0.316164,...,1.485180,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
567005,99999,113369.0,2118-01-01,Nursing,Nursing Transfer Note,0.425454,0.510033,-0.442609,0.970444,-0.316164,...,-0.464806,0.191607,-0.210975,0.638653,-0.370967,2.830315,-0.446205,-0.429672,-0.251045,-0.724288
567006,99999,113369.0,2118-01-01,Physician,Intensivist Note,-0.520320,0.365397,-0.007890,1.574738,0.572302,...,0.872858,0.077255,-0.210975,0.474349,2.767061,3.702053,-0.065075,0.516563,-0.251045,-0.724288


In [19]:
agg_notes['HADM_ID'] = agg_notes['HADM_ID'].astype(int)
agg_notes['SUBJECT_ID'] = agg_notes['SUBJECT_ID'].astype(int)
agg_notes['CHARTDATE'] = pd.to_datetime(agg_notes['CHARTDATE']).dt.date

In [20]:
data_multi = data_multi.reset_index(drop=True)
data_multi['HADM_ID'] = data_multi['HADM_ID'].astype(int)
data_multi['SUBJECT_ID'] = data_multi['SUBJECT_ID'].astype(int)
data_multi = data_multi.dropna(subset= ["CHARTTIME"])
data_multi['CHARTTIME'] = pd.to_datetime(data_multi['CHARTTIME'], errors='coerce')
data_multi['CHARTDATE'] = data_multi['CHARTTIME'].dt.date
data_multi

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,CHARTTIME,LOS,GENDER,AGE,Heart Rate,O2 Saturation,Respiratory Rate,...,Weight,GCS Total,Lactic Acid,FiO2,PaO2,ADMISSION_TYPE,INSURANCE,HOSPITAL_EXPIRE_FLAG,AgeGroup,CHARTDATE
0,3,145834,211552,2101-10-21 11:00:00,6.0646,1,77.0,88.0,98.0,9.0,...,269.240000,15.0,1.601220,0.40,80.772097,1,2,0,61-80,2101-10-21
1,3,145834,211552,2101-10-24 21:00:00,6.0646,1,77.0,86.0,98.0,17.0,...,269.240000,15.0,1.549035,0.40,80.772097,1,2,0,61-80,2101-10-24
2,3,145834,211552,2101-10-24 20:00:00,6.0646,1,77.0,85.0,99.0,17.0,...,269.240000,15.0,0.730116,0.40,105.000000,1,2,0,61-80,2101-10-24
3,3,145834,211552,2101-10-24 19:00:00,6.0646,1,77.0,84.0,100.0,18.0,...,269.240000,15.0,1.471546,0.40,105.000000,1,2,0,61-80,2101-10-24
4,3,145834,211552,2101-10-24 18:00:00,6.0646,1,77.0,88.0,100.0,17.0,...,269.240000,15.0,0.612001,0.40,105.000000,1,2,0,61-80,2101-10-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3072757,99999,113369,246512,2117-12-31 14:00:00,1.1242,0,64.0,98.0,97.0,18.0,...,75.416445,4.0,0.764668,0.21,69.524495,0,2,0,61-80,2117-12-31
3072758,99999,113369,246512,2117-12-31 13:00:00,1.1242,0,64.0,94.0,97.0,15.0,...,75.416445,4.0,0.823122,0.21,69.524495,0,2,0,61-80,2117-12-31
3072759,99999,113369,246512,2118-01-01 13:00:00,1.1242,0,64.0,86.0,97.0,15.0,...,75.416445,4.0,1.818850,0.21,69.524495,0,2,0,61-80,2118-01-01
3072760,99999,113369,246512,2118-01-01 00:00:00,1.1242,0,64.0,100.0,97.0,21.0,...,75.416445,4.0,1.773556,0.21,69.524495,0,2,0,61-80,2118-01-01


In [22]:
data_multi = data_multi.drop(columns = ["AgeGroup","CHARTTIME"])

In [23]:
df = data_multi.merge(agg_notes, on=["HADM_ID","SUBJECT_ID","CHARTDATE"], how = 'inner')

In [24]:
df = df.drop(columns = ["SUBJECT_ID","HADM_ID","ICUSTAY_ID",'CHARTDATE'])

In [25]:
df = df.dropna()

In [26]:
df

,LOS,GENDER,AGE,Heart Rate,O2 Saturation,Respiratory Rate,Temperature (C),Mean Blood Pressure,Systolic Blood Pressure,Diastolic Blood Pressure,...,tfidf_90,tfidf_91,tfidf_92,tfidf_93,tfidf_94,tfidf_95,tfidf_96,tfidf_97,tfidf_98,tfidf_99
0,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
1,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,-0.464806,0.517090,-0.210975,1.106314,-0.370967,-0.358055,-0.446205,-0.429672,2.381485,-0.724288
2,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,1.539497,0.076552,-0.210975,0.473340,4.330931,2.345726,-0.446205,8.077132,-0.251045,0.299629
3,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,0.736311,1.742810,-0.210975,-0.449029,-0.370967,-0.358055,0.580470,-0.429672,-0.251045,0.656316
4,6.0646,1,77.0,86.0,98.0,17.0,36.933333,79.041667,115.125000,61.000000,...,0.494767,1.278629,-0.210975,0.875740,3.005628,-0.358055,-0.446205,0.927890,-0.251045,-0.724288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7260219,1.1242,0,64.0,100.0,97.0,21.0,36.587685,81.650256,92.062569,76.444099,...,-0.464806,0.191607,-0.210975,0.638653,-0.370967,2.830315,-0.446205,-0.429672,-0.251045,-0.724288
7260220,1.1242,0,64.0,100.0,97.0,21.0,36.587685,81.650256,92.062569,76.444099,...,0.872858,0.077255,-0.210975,0.474349,2.767061,3.702053,-0.065075,0.516563,-0.251045,-0.724288
7260221,1.1242,0,64.0,83.0,97.0,20.0,36.821877,88.750504,111.334241,77.458635,...,1.485180,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
7260222,1.1242,0,64.0,83.0,97.0,20.0,36.821877,88.750504,111.334241,77.458635,...,-0.464806,0.191607,-0.210975,0.638653,-0.370967,2.830315,-0.446205,-0.429672,-0.251045,-0.724288


In [27]:
df_small = df.sample(frac=0.3, random_state=42)
df_small = df_small.sort_index().reset_index(drop = True)
df_small

,LOS,GENDER,AGE,Heart Rate,O2 Saturation,Respiratory Rate,Temperature (C),Mean Blood Pressure,Systolic Blood Pressure,Diastolic Blood Pressure,...,tfidf_90,tfidf_91,tfidf_92,tfidf_93,tfidf_94,tfidf_95,tfidf_96,tfidf_97,tfidf_98,tfidf_99
0,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
1,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,1.539497,0.076552,-0.210975,0.473340,4.330931,2.345726,-0.446205,8.077132,-0.251045,0.299629
2,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,0.736311,1.742810,-0.210975,-0.449029,-0.370967,-0.358055,0.580470,-0.429672,-0.251045,0.656316
3,6.0646,1,77.0,86.0,98.0,17.0,36.933333,79.041667,115.125000,61.000000,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,1.014802
4,6.0646,1,77.0,84.0,100.0,18.0,36.800000,78.458333,117.375000,59.000000,...,0.494767,1.278629,-0.210975,0.875740,3.005628,-0.358055,-0.446205,0.927890,-0.251045,-0.724288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2178062,1.1242,0,64.0,101.0,97.0,20.0,36.982697,92.501582,118.283516,79.610615,...,1.224191,0.246046,-0.210975,0.716871,-0.370967,4.768429,-0.446205,0.765088,-0.251045,-0.724288
2178063,1.1242,0,64.0,94.0,97.0,15.0,37.030493,91.407209,115.528930,79.346348,...,1.224191,0.246046,-0.210975,0.716871,-0.370967,4.768429,-0.446205,0.765088,-0.251045,-0.724288
2178064,1.1242,0,64.0,100.0,97.0,21.0,36.587685,81.650256,92.062569,76.444099,...,1.485180,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
2178065,1.1242,0,64.0,100.0,97.0,21.0,36.587685,81.650256,92.062569,76.444099,...,0.872858,0.077255,-0.210975,0.474349,2.767061,3.702053,-0.065075,0.516563,-0.251045,-0.724288


In [28]:
print(list(df_small.columns))

['LOS', 'GENDER', 'AGE', 'Heart Rate', 'O2 Saturation', 'Respiratory Rate', 'Temperature (C)', 'Mean Blood Pressure', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'WBC', 'pH', 'Glucose', 'Sodium', 'Potassium', 'Creatinine', 'BUN', 'Hematocrit', 'Height', 'Weight', 'GCS Total', 'Lactic Acid', 'FiO2', 'PaO2', 'ADMISSION_TYPE', 'INSURANCE', 'HOSPITAL_EXPIRE_FLAG', 'CATEGORY', 'DESCRIPTION', 'TEXT', 'tfidf_0', 'tfidf_1', 'tfidf_2', 'tfidf_3', 'tfidf_4', 'tfidf_5', 'tfidf_6', 'tfidf_7', 'tfidf_8', 'tfidf_9', 'tfidf_10', 'tfidf_11', 'tfidf_12', 'tfidf_13', 'tfidf_14', 'tfidf_15', 'tfidf_16', 'tfidf_17', 'tfidf_18', 'tfidf_19', 'tfidf_20', 'tfidf_21', 'tfidf_22', 'tfidf_23', 'tfidf_24', 'tfidf_25', 'tfidf_26', 'tfidf_27', 'tfidf_28', 'tfidf_29', 'tfidf_30', 'tfidf_31', 'tfidf_32', 'tfidf_33', 'tfidf_34', 'tfidf_35', 'tfidf_36', 'tfidf_37', 'tfidf_38', 'tfidf_39', 'tfidf_40', 'tfidf_41', 'tfidf_42', 'tfidf_43', 'tfidf_44', 'tfidf_45', 'tfidf_46', 'tfidf_47', 'tfidf_48', 'tfidf_49', '

In [29]:
df_small.drop(columns = "TEXT", axis = 1)

,LOS,GENDER,AGE,Heart Rate,O2 Saturation,Respiratory Rate,Temperature (C),Mean Blood Pressure,Systolic Blood Pressure,Diastolic Blood Pressure,...,tfidf_90,tfidf_91,tfidf_92,tfidf_93,tfidf_94,tfidf_95,tfidf_96,tfidf_97,tfidf_98,tfidf_99
0,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
1,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,1.539497,0.076552,-0.210975,0.473340,4.330931,2.345726,-0.446205,8.077132,-0.251045,0.299629
2,6.0646,1,77.0,88.0,98.0,9.0,37.000000,78.000000,114.000000,62.000000,...,0.736311,1.742810,-0.210975,-0.449029,-0.370967,-0.358055,0.580470,-0.429672,-0.251045,0.656316
3,6.0646,1,77.0,86.0,98.0,17.0,36.933333,79.041667,115.125000,61.000000,...,-0.464806,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,1.014802
4,6.0646,1,77.0,84.0,100.0,18.0,36.800000,78.458333,117.375000,59.000000,...,0.494767,1.278629,-0.210975,0.875740,3.005628,-0.358055,-0.446205,0.927890,-0.251045,-0.724288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2178062,1.1242,0,64.0,101.0,97.0,20.0,36.982697,92.501582,118.283516,79.610615,...,1.224191,0.246046,-0.210975,0.716871,-0.370967,4.768429,-0.446205,0.765088,-0.251045,-0.724288
2178063,1.1242,0,64.0,94.0,97.0,15.0,37.030493,91.407209,115.528930,79.346348,...,1.224191,0.246046,-0.210975,0.716871,-0.370967,4.768429,-0.446205,0.765088,-0.251045,-0.724288
2178064,1.1242,0,64.0,100.0,97.0,21.0,36.587685,81.650256,92.062569,76.444099,...,1.485180,-0.565398,-0.210975,-0.449029,-0.370967,-0.358055,-0.446205,-0.429672,-0.251045,-0.724288
2178065,1.1242,0,64.0,100.0,97.0,21.0,36.587685,81.650256,92.062569,76.444099,...,0.872858,0.077255,-0.210975,0.474349,2.767061,3.702053,-0.065075,0.516563,-0.251045,-0.724288


In [30]:
df_small
df_small.to_parquet("/content/drive/MyDrive/ERP/df_merged_3M.parquet", index = False)